# Demand Forecasting & Business Analytics — Phase 1

This notebook is the research and forecasting engine for a Python-only project.

**Supported input:** CSV, XLSX, XLS  
**Target:** selected by the user  
**Series types:** one overall series or multiple products/stores/regions  
**Frequency:** automatic initial suggestion for daily, weekly, or monthly data  
**Forecast horizon:** one calendar year beyond the last observation

> Important: automatic column/frequency detection is a suggestion, not a guarantee. Review the configuration and data preview before running the modelling cells. Forecast quality depends on the amount and quality of historical data.

## 1. Install dependencies

Run this cell once in the notebook environment. If installation finishes, restart the kernel only if Jupyter asks you to.

In [1]:
%pip install -U pandas numpy matplotlib seaborn plotly scikit-learn statsmodels openpyxl xlrd joblib nbformat

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

Set `FILE_PATH` to your dataset. Start with `TARGET_COLUMN = None` to inspect columns, then set it to the exact column you want to forecast. Set `DATE_COLUMN` explicitly if automatic detection picks the wrong column.

Use `GROUP_COLUMNS` for independent series such as `["Store", "Product"]`. Leave it empty for one overall series. Do not include a unique transaction/order ID as a group column.

In [2]:
from pathlib import Path
import warnings
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.ar_model import AutoReg

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
FILE_PATH = (Path("..") / "data" / "Walmart.csv")
DATE_COLUMN = "Date"
TARGET_COLUMN = "Weekly_Sales"
GROUP_COLUMNS = ["Store"]
FREQUENCY = "W"
FORECAST_ONE_CALENDAR_YEAR = True
MAX_GROUPS_TO_MODEL = 20
RANDOM_STATE = 42

print("Notebook configuration loaded.")

Notebook configuration loaded.


## 3. Load the dataset

For the first notebook run, place a file in `data/` and update `FILE_PATH`. Excel files are supported. This cell also prints a small preview and basic metadata.

In [3]:
def load_sales_file(file_path: Path) -> pd.DataFrame:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(
            f"Could not find {file_path.resolve()}. Put your sales file in the data folder "
            "or update FILE_PATH in the configuration cell."
        )
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

if Path(FILE_PATH).is_dir():
    candidates = [
        Path(FILE_PATH) / "data" / "Walmart.csv",
        Path.cwd() / "data" / "Walmart.csv",
        Path.cwd().parent / "data" / "Walmart.csv",
    ]
    FILE_PATH = next((path for path in candidates if path.is_file()), FILE_PATH)

raw_df = load_sales_file(FILE_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]
print(f"Rows: {len(raw_df):,} | Columns: {len(raw_df.columns)}")
display(raw_df.head())
display(pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
    "unique_count": raw_df.nunique(dropna=True)
}))

Rows: 6,435 | Columns: 8


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


,dtype,missing_count,missing_percent,unique_count
Store,int64,0,0.0,45
Date,str,0,0.0,143
Weekly_Sales,float64,0,0.0,6435
Holiday_Flag,int64,0,0.0,2
Temperature,float64,0,0.0,3528
Fuel_Price,float64,0,0.0,892
CPI,float64,0,0.0,2145
Unemployment,float64,0,0.0,349


## 4. Suggest date and numeric target columns

Review these suggestions. For reliable forecasting, the target should be a numeric quantity such as units sold, demand, or revenue—not a product code, ID, or date.

In [4]:
DATE_HINTS = ("date", "day", "time", "period", "week", "month", "timestamp")
TARGET_HINTS = ("sales", "quantity", "units", "demand", "revenue", "amount", "orders", "target")

def suggest_date_column(df):
    candidates = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        name_score = any(h in col.lower() for h in DATE_HINTS)
        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        valid_ratio = parsed.notna().mean()
        if name_score or valid_ratio >= 0.8:
            candidates.append((col, valid_ratio, name_score))
    candidates.sort(key=lambda x: (x[2], x[1]), reverse=True)
    return candidates

def suggest_target_columns(df):
    rows = []
    for col in df.columns:
        numeric = pd.to_numeric(df[col], errors="coerce")
        ratio = numeric.notna().mean()
        if ratio >= 0.8 and not any(h in col.lower() for h in ("id", "code", "zip", "postal", "year")):
            hint = any(h in col.lower() for h in TARGET_HINTS)
            rows.append((col, round(ratio, 3), hint, numeric.nunique()))
    rows.sort(key=lambda x: (x[2], x[1], x[3]), reverse=True)
    return rows

print("Date column suggestions (column, parseable ratio, name hint):")
display(pd.DataFrame(suggest_date_column(raw_df), columns=["column", "parseable_ratio", "name_hint"]))
print("Numeric target suggestions (column, numeric_ratio, name_hint, unique_values):")
display(pd.DataFrame(suggest_target_columns(raw_df), columns=["column", "numeric_ratio", "name_hint", "unique_values"]))

print("\nAvailable columns:", list(raw_df.columns))

Date column suggestions (column, parseable ratio, name hint):


,column,parseable_ratio,name_hint
0,Date,1.0,True


Numeric target suggestions (column, numeric_ratio, name_hint, unique_values):


,column,numeric_ratio,name_hint,unique_values
0,Weekly_Sales,1.0,True,6435
1,Temperature,1.0,False,3528
2,CPI,1.0,False,2145
3,Fuel_Price,1.0,False,892
4,Unemployment,1.0,False,349
5,Store,1.0,False,45



Available columns: ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']


## 5. Configure the columns and validate the data

After reviewing the suggestions, set `DATE_COLUMN`, `TARGET_COLUMN`, and `GROUP_COLUMNS` in the configuration cell, then rerun that cell and the cells below it. The fallback below uses the strongest suggestion when a value is still `None`.

In [5]:
if DATE_COLUMN is None:
    date_candidates = suggest_date_column(raw_df)
    if not date_candidates:
        raise ValueError("No likely date column found. Set DATE_COLUMN manually.")
    DATE_COLUMN = date_candidates[0][0]

if TARGET_COLUMN is None:
    target_candidates = suggest_target_columns(raw_df)
    target_candidates = [candidate for candidate in target_candidates if candidate[0] != DATE_COLUMN]
    if not target_candidates:
        raise ValueError("No likely numeric target column different from DATE_COLUMN was found. Set TARGET_COLUMN manually.")
    TARGET_COLUMN = target_candidates[0][0]

if DATE_COLUMN == TARGET_COLUMN:
    raise ValueError("DATE_COLUMN and TARGET_COLUMN must be different.")

missing_columns = [c for c in [DATE_COLUMN, TARGET_COLUMN, *GROUP_COLUMNS] if c not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Configured columns not found: {missing_columns}")

df = raw_df.copy()
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce", dayfirst=True)
df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
before = len(df)
df = df.dropna(subset=[DATE_COLUMN, TARGET_COLUMN]).copy()
df = df.sort_values(DATE_COLUMN)
print(f"Selected date: {DATE_COLUMN} | target: {TARGET_COLUMN} | groups: {GROUP_COLUMNS or 'Overall'}")
print(f"Removed {before - len(df):,} rows with invalid/missing date or target.")
print(f"Date range: {df[DATE_COLUMN].min()} to {df[DATE_COLUMN].max()}")
display(df.head())

Selected date: Date | target: Weekly_Sales | groups: ['Store']
Removed 0 rows with invalid/missing date or target.
Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1287,10,2010-02-05,2193048.75,0,54.34,2.962,126.442065,9.765
5148,37,2010-02-05,536006.73,0,45.97,2.572,209.852966,8.554
2288,17,2010-02-05,789036.02,0,23.11,2.666,126.442065,6.548
4147,30,2010-02-05,465108.52,0,39.05,2.572,210.752605,8.324


## 6. Detect daily, weekly, or monthly frequency

This is a heuristic. It uses the median gap between unique dates and prints a suggested frequency. Irregular or sparse data can fool automatic detection, so inspect the date spacing and override `FREQUENCY` when needed.

- `D`: daily
- `W`: weekly, weeks ending Sunday
- `MS`: monthly, month start

In [6]:
def detect_frequency(dates: pd.Series) -> str:
    unique_dates = pd.Series(pd.to_datetime(dates).dropna().unique()).sort_values()
    if len(unique_dates) < 3:
        return "MS"
    gaps = unique_dates.diff().dropna().dt.total_seconds() / 86400
    median_gap = float(gaps.median())
    if median_gap <= 2.0:
        return "D"
    if median_gap <= 10.0:
        return "W"
    return "MS"

detected_frequency = detect_frequency(df[DATE_COLUMN])
active_frequency = detected_frequency if FREQUENCY == "auto" else FREQUENCY
if active_frequency not in {"D", "W", "MS"}:
    raise ValueError("FREQUENCY must be 'auto', 'D', 'W', or 'MS'.")

print(f"Suggested frequency: {detected_frequency}")
print(f"Frequency used: {active_frequency}")
print("Unique dates:", df[DATE_COLUMN].nunique())
print("Median gap in days:", round(df[DATE_COLUMN].sort_values().drop_duplicates().diff().dropna().dt.days.median(), 2))

Suggested frequency: W
Frequency used: W
Unique dates: 143
Median gap in days: 7.0


## 7. Aggregate data into a regular time series

Repeated records on the same date are summed. If groups are configured, each product/store combination becomes its own series. Missing periods are filled with zero **only as an initial assumption**; if missing dates mean “not recorded” rather than “zero sales,” change that policy before trusting forecasts.

In [7]:
def make_period_index(dates: pd.Series, freq: str) -> pd.DatetimeIndex:
    start = pd.to_datetime(dates.min())
    end = pd.to_datetime(dates.max())
    if freq == "D":
        return pd.date_range(start=start.normalize(), end=end.normalize(), freq="D")
    if freq == "W":
        return pd.date_range(start=start, end=end, freq="W-SUN")
    return pd.date_range(start=start.to_period("M").to_timestamp(), end=end.to_period("M").to_timestamp(), freq="MS")

def build_series_table(frame, date_col, target_col, group_cols, freq):
    work = frame[[date_col, target_col, *group_cols]].copy()
    if freq == "D":
        work["_period"] = work[date_col].dt.floor("D")
    elif freq == "W":
        work["_period"] = work[date_col].dt.to_period("W-SUN").dt.end_time.dt.normalize()
    else:
        work["_period"] = work[date_col].dt.to_period("M").dt.to_timestamp()
    keys = [*group_cols, "_period"]
    grouped = work.groupby(keys, dropna=False, as_index=False)[target_col].sum()
    series_list = []
    groupby_arg = group_cols[0] if len(group_cols) == 1 else group_cols
    if group_cols:
        grouped_groups = grouped.groupby(groupby_arg, dropna=False, sort=False)
        for group_key, part in grouped_groups:
            group_values = group_key if isinstance(group_key, tuple) else (group_key,)
            label = " | ".join(f"{col}={val}" for col, val in zip(group_cols, group_values))
            part = part.set_index("_period")[target_col].sort_index()
            idx = make_period_index(part.index.to_series(), freq)
            y = part.reindex(idx, fill_value=0).astype(float)
            y.name = target_col
            series_list.append({"label": label, "group_values": dict(zip(group_cols, group_values)), "y": y})
    else:
        part = grouped.set_index("_period")[target_col].sort_index()
        idx = make_period_index(part.index.to_series(), freq)
        y = part.reindex(idx, fill_value=0).astype(float)
        y.name = target_col
        series_list.append({"label": "Overall", "group_values": {}, "y": y})
    return series_list

series_data = build_series_table(df, DATE_COLUMN, TARGET_COLUMN, GROUP_COLUMNS, active_frequency)
print(f"Prepared {len(series_data)} time series.")
for item in series_data[:5]:
    print(item["label"], "| periods:", len(item["y"]), "| range:", item["y"].index.min(), "to", item["y"].index.max())
if len(series_data) > MAX_GROUPS_TO_MODEL:
    print(f"WARNING: {len(series_data)} groups found. The modelling step will use the first {MAX_GROUPS_TO_MODEL} groups for this initial run.")

Prepared 45 time series.
Store=1 | periods: 143 | range: 2010-02-07 00:00:00 to 2012-10-28 00:00:00
Store=2 | periods: 143 | range: 2010-02-07 00:00:00 to 2012-10-28 00:00:00
Store=3 | periods: 143 | range: 2010-02-07 00:00:00 to 2012-10-28 00:00:00
Store=4 | periods: 143 | range: 2010-02-07 00:00:00 to 2012-10-28 00:00:00
Store=5 | periods: 143 | range: 2010-02-07 00:00:00 to 2012-10-28 00:00:00


## 8. Exploratory data analysis

These charts are the first building blocks of the Power BI-style dashboard. Inspect outliers and long periods of zero sales before modelling.

In [8]:
display(df.describe(include="all").T)
fig = px.histogram(df, x=TARGET_COLUMN, nbins=40, title=f"Distribution of {TARGET_COLUMN}")
fig.show()

if GROUP_COLUMNS:
    group_summary = df.groupby(GROUP_COLUMNS, dropna=False)[TARGET_COLUMN].sum().sort_values(ascending=False).head(15).reset_index()
    display(group_summary)
    fig = px.bar(group_summary, x=TARGET_COLUMN, y=GROUP_COLUMNS[-1], orientation="h",
                 title=f"Top groups by total {TARGET_COLUMN}")
    fig.show()

fig = px.line(df.sort_values(DATE_COLUMN), x=DATE_COLUMN, y=TARGET_COLUMN,
              title=f"Raw {TARGET_COLUMN} over time")
fig.show()

for item in series_data[:min(5, len(series_data))]:
    fig = px.line(x=item["y"].index, y=item["y"].values,
                  labels={"x": "Date", "y": TARGET_COLUMN},
                  title=f"Aggregated time series — {item['label']}")
    fig.show()

,count,mean,min,25%,50%,75%,max,std
Store,6435.0,23.0,1.0,12.0,23.0,34.0,45.0,12.988182
Date,6435,2011-06-17 00:00:00,2010-02-05 00:00:00,2010-10-08 00:00:00,2011-06-17 00:00:00,2012-02-24 00:00:00,2012-10-26 00:00:00,NaN
Weekly_Sales,6435.0,1046964.877562,209986.25,553350.105,960746.04,1420158.66,3818686.45,564366.622054
Holiday_Flag,6435.0,0.06993,0.0,0.0,0.0,0.0,1.0,0.255049
Temperature,6435.0,60.663782,-2.06,47.46,62.67,74.94,100.14,18.444933
Fuel_Price,6435.0,3.358607,2.472,2.933,3.445,3.735,4.468,0.45902
CPI,6435.0,171.578394,126.064,131.735,182.616521,212.743293,227.232807,39.356712
Unemployment,6435.0,7.999151,3.879,6.891,7.874,8.622,14.313,1.875885


,Store,Weekly_Sales
0,20,3.013978e+08
1,4,2.995440e+08
2,14,2.889999e+08
3,13,2.865177e+08
4,2,2.753824e+08
5,10,2.716177e+08
6,27,2.538559e+08
7,6,2.237561e+08
8,1,2.224028e+08
9,39,2.074455e+08


## 9. Forecasting utilities and evaluation metrics

The notebook compares a seasonal-naive baseline, Exponential Smoothing, and AutoReg when the history is long enough. It uses a chronological holdout: the most recent observations are reserved for evaluation. This is more appropriate for time series than random train/test splitting.

The first version uses one holdout per series for clarity. Later we can upgrade this to rolling-origin backtesting and broader model families.

In [9]:
def seasonal_periods(freq):
    return {"D": 7, "W": 52, "MS": 12}[freq]

def forecast_index(last_date, freq, horizon):
    if freq == "D":
        return pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq="D")
    if freq == "W":
        return pd.date_range(last_date + pd.Timedelta(days=7), periods=horizon, freq="W-SUN")
    return pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")

def calendar_year_horizon(last_date, freq):
    end_date = pd.Timestamp(last_date) + pd.DateOffset(years=1)
    if freq == "D":
        return max(1, len(pd.date_range(pd.Timestamp(last_date) + pd.Timedelta(days=1), end_date, freq="D")))
    if freq == "W":
        return max(1, len(pd.date_range(pd.Timestamp(last_date) + pd.Timedelta(days=7), end_date, freq="W-SUN")))
    return max(1, len(pd.date_range(pd.Timestamp(last_date).to_period("M").to_timestamp() + pd.offsets.MonthBegin(1),
                                    end_date.to_period("M").to_timestamp(), freq="MS")))

def seasonal_naive_forecast(train, horizon, period):
    values = np.asarray(train, dtype=float)
    if len(values) >= period:
        pattern = values[-period:]
    else:
        pattern = np.repeat(values[-1] if len(values) else 0.0, period)
    return np.resize(pattern, horizon).astype(float)

def smape(y_true, y_pred):
    a, b = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = np.abs(a) + np.abs(b)
    valid = denom != 0
    return float(np.mean(2 * np.abs(a[valid] - b[valid]) / denom[valid]) * 100) if valid.any() else 0.0

def wape(y_true, y_pred):
    a, b = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = np.sum(np.abs(a))
    return float(np.sum(np.abs(a - b)) / denom * 100) if denom else np.nan

def evaluate_candidate_models(y, freq):
    y = pd.Series(y).astype(float).dropna()
    period = seasonal_periods(freq)
    minimum = max(8, period * 2)
    if len(y) < minimum + 3:
        return pd.DataFrame([{
            "model": "Seasonal Naive",
            "MAE": np.nan, "RMSE": np.nan, "sMAPE_%": np.nan, "WAPE_%": np.nan,
            "status": f"Insufficient history for robust holdout comparison (need about {minimum + 3}+ periods)."
        }]), None

    horizon = min(max(3, period), max(1, len(y) // 5))
    train, test = y.iloc[:-horizon], y.iloc[-horizon:]
    candidates = {"Seasonal Naive": seasonal_naive_forecast(train, len(test), period)}
    if len(train) >= max(10, period * 2):
        try:
            seasonal = period if len(train) >= period * 2 else None
            model = ExponentialSmoothing(
                train, trend="add", seasonal="add" if seasonal else None,
                seasonal_periods=seasonal, initialization_method="estimated"
            ).fit(optimized=True)
            candidates["Exponential Smoothing"] = np.asarray(model.forecast(len(test)))
        except Exception:
            pass
    try:
        lags = min(period, max(1, len(train) // 4))
        if len(train) > lags + 5:
            model = AutoReg(train, lags=lags, old_names=False, trend="ct").fit()
            candidates["AutoReg"] = np.asarray(model.predict(start=len(train), end=len(train) + len(test) - 1, dynamic=False))
    except Exception:
        pass

    rows = []
    for name, pred in candidates.items():
        pred = np.asarray(pred, dtype=float)
        rows.append({
            "model": name,
            "MAE": mean_absolute_error(test, pred),
            "RMSE": math.sqrt(mean_squared_error(test, pred)),
            "sMAPE_%": smape(test, pred),
            "WAPE_%": wape(test, pred),
            "status": "evaluated"
        })
    results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
    return results, {"train": train, "test": test, "predictions": candidates, "horizon": horizon}

print("Forecasting functions are ready.")

Forecasting functions are ready.


## 10. Compare models for the selected series

For a large number of groups, this initial notebook limits modelling to `MAX_GROUPS_TO_MODEL`. Increase the limit after checking runtime and data quality.

In [10]:
series_to_model = series_data[:MAX_GROUPS_TO_MODEL]
evaluation_results = []
holdout_details = {}

for item in series_to_model:
    results, details = evaluate_candidate_models(item["y"], active_frequency)
    results.insert(0, "series", item["label"])
    evaluation_results.append(results)
    holdout_details[item["label"]] = details

evaluation_df = pd.concat(evaluation_results, ignore_index=True) if evaluation_results else pd.DataFrame()
display(evaluation_df)

,series,model,MAE,RMSE,sMAPE_%,WAPE_%,status
0,Store=1,Exponential Smoothing,54194.251757,70217.777768,3.389041,3.452053,evaluated
1,Store=1,Seasonal Naive,67765.999286,85613.270570,4.408895,4.316543,evaluated
2,Store=2,Exponential Smoothing,91213.683363,106863.014552,4.746662,4.827620,evaluated
3,Store=2,Seasonal Naive,92487.483214,112938.753791,5.024131,4.895037,evaluated
4,Store=3,Exponential Smoothing,17280.255898,21041.734270,4.064292,4.143021,evaluated
5,Store=3,Seasonal Naive,36146.581429,38964.435005,9.066795,8.666309,evaluated
6,Store=4,Exponential Smoothing,69300.533836,88679.248343,3.171457,3.219784,evaluated
7,Store=4,Seasonal Naive,96366.678571,111569.623121,4.588635,4.477309,evaluated
8,Store=5,Exponential Smoothing,15019.062734,18603.878891,4.519205,4.582655,evaluated
9,Store=5,Seasonal Naive,23967.034643,26909.856880,7.588718,7.312884,evaluated


## 11. Fit the selected model and forecast one calendar year

The best model is chosen by holdout MAE when evaluation is possible. When the history is too short, the notebook falls back to seasonal naive and clearly marks the limitation. This is a baseline research implementation—not a promise of accurate forecasts for every dataset.

In [11]:
def fit_and_forecast(y, freq, horizon):
    y = pd.Series(y).astype(float).dropna()
    period = seasonal_periods(freq)
    results, _ = evaluate_candidate_models(y, freq)
    evaluated = results[results["status"] == "evaluated"].copy() if "status" in results else pd.DataFrame()
    selected = evaluated.sort_values("MAE").iloc[0]["model"] if not evaluated.empty else "Seasonal Naive"
    note = "Selected by chronological holdout MAE." if not evaluated.empty else "Short history: using baseline; forecast confidence is limited."

    if selected == "Exponential Smoothing":
        try:
            seasonal = period if len(y) >= period * 2 else None
            fitted = ExponentialSmoothing(
                y, trend="add", seasonal="add" if seasonal else None,
                seasonal_periods=seasonal, initialization_method="estimated"
            ).fit(optimized=True)
            prediction = np.asarray(fitted.forecast(horizon), dtype=float)
        except Exception:
            selected = "Seasonal Naive"
            prediction = seasonal_naive_forecast(y, horizon, period)
            note = "Exponential Smoothing failed on full history; used baseline instead."
    elif selected == "AutoReg":
        try:
            lags = min(period, max(1, len(y) // 4))
            fitted = AutoReg(y, lags=lags, old_names=False, trend="ct").fit()
            prediction = np.asarray(fitted.predict(start=len(y), end=len(y) + horizon - 1, dynamic=False), dtype=float)
        except Exception:
            selected = "Seasonal Naive"
            prediction = seasonal_naive_forecast(y, horizon, period)
            note = "AutoReg failed on full history; used baseline instead."
    else:
        prediction = seasonal_naive_forecast(y, horizon, period)

    prediction = np.maximum(prediction, 0)
    future_index = forecast_index(y.index[-1], freq, horizon)
    forecast_df = pd.DataFrame({
        "date": future_index,
        "prediction": prediction,
        "selected_model": selected,
        "note": note
    })
    return forecast_df

forecast_frames = []
for item in series_to_model:
    horizon = calendar_year_horizon(item["y"].index[-1], active_frequency) if FORECAST_ONE_CALENDAR_YEAR else 12
    one_forecast = fit_and_forecast(item["y"], active_frequency, horizon)
    one_forecast["series"] = item["label"]
    for col, val in item["group_values"].items():
        one_forecast[col] = val
    forecast_frames.append(one_forecast)

forecast_df = pd.concat(forecast_frames, ignore_index=True) if forecast_frames else pd.DataFrame()
display(forecast_df.head(20))

,date,prediction,selected_model,note,series,Store
0,2012-11-04,1.690366e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
1,2012-11-11,1.618977e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
2,2012-11-18,1.594760e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
3,2012-11-25,2.073036e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
4,2012-12-02,1.653538e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
5,2012-12-09,1.811618e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
6,2012-12-16,1.983270e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
7,2012-12-23,2.450583e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
8,2012-12-30,1.503144e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1
9,2013-01-06,1.573173e+06,Exponential Smoothing,Selected by chronological holdout MAE.,Store=1,1


## 12. Visualize forecast results

The historical line and future forecast are shown together. This initial plot displays the first modelled series; the later dashboard will add interactive group selection and uncertainty intervals.

In [12]:
if series_to_model and not forecast_df.empty:
    first = series_to_model[0]
    future = forecast_df[forecast_df["series"] == first["label"]]
    hist_df = pd.DataFrame({"date": first["y"].index, "value": first["y"].values, "kind": "Historical"})
    pred_df = pd.DataFrame({"date": future["date"], "value": future["prediction"], "kind": "Forecast"})
    plot_df = pd.concat([hist_df, pred_df], ignore_index=True)
    fig = px.line(plot_df, x="date", y="value", color="kind",
                  title=f"Historical and forecasted {TARGET_COLUMN} — {first['label']}")
    fig.show()

    monthly_or_period_summary = forecast_df.groupby("date", as_index=False)["prediction"].sum()
    fig = px.bar(monthly_or_period_summary, x="date", y="prediction",
                 title=f"Forecasted {TARGET_COLUMN} by future period (modelled groups)")
    fig.show()

## 13. Export forecast and model evaluation

These files will later be returned by the backend or offered as dashboard downloads.

In [13]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

forecast_path = OUTPUT_DIR / "demand_forecast.csv"
evaluation_path = OUTPUT_DIR / "model_evaluation.csv"
forecast_df.to_csv(forecast_path, index=False)
evaluation_df.to_csv(evaluation_path, index=False)

print("Saved:", forecast_path.resolve())
print("Saved:", evaluation_path.resolve())

Saved: C:\Users\dell\Desktop\DemandForecasting\notebooks\outputs\demand_forecast.csv
Saved: C:\Users\dell\Desktop\DemandForecasting\notebooks\outputs\model_evaluation.csv


## 14. Phase 1 validation checklist

Before we move to the FastAPI backend, verify:

1. The selected date and target columns are correct.
2. Frequency detection matches the real dataset.
3. Grouping produces the expected products/stores.
4. Missing periods are interpreted correctly (zero vs unknown).
5. The holdout results are visible and no future data leaked into training.
6. Forecast dates extend one calendar year beyond the final observation.
7. The exported CSV files open correctly.

**Known first-version limitations:** frequency detection is heuristic; missing periods default to zero; the notebook caps the number of groups; prediction intervals are not yet implemented; model selection uses one chronological holdout rather than rolling-origin backtesting. We will address these before presenting this as a production-quality tool.